# Wind Power Predictive Modeling - Exploratory Data Analysis (EDA)

This notebook explores the preprocessed meteorological data to understand spatial, temporal, and feature-related patterns concerning Wind Power Density across Indian states.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
DATA_PATH = r'../Fetch and Preprocessing (Wind)/India_Renewable_Energy_MASTER_DATASET_Calculated.csv'
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
print(f"Dataset loaded successfully with {len(df)} rows and {df.shape[1]} columns.")

## 1. Spatial Analysis: State-wise Wind Power Potential

In [ ]:
state_avg_power = df.groupby('State')['Wind_Power_Density'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 8))
sns.barplot(x=state_avg_power.values, y=state_avg_power.index, palette='viridis')
plt.title('Average Wind Power Density by State (2015-2024)', fontsize=16)
plt.xlabel('Average Wind Power Density (W/m^2)', fontsize=12)
plt.ylabel('State / Union Territory', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 2. Temporal Analysis: Seasonality and Autocorrelation

In [ ]:
df['Month'] = df['Date'].dt.month

plt.figure(figsize=(12, 6))
sns.boxplot(x='Month', y='Wind_Power_Density', data=df, palette='Set3')
plt.title('Monthly Distribution of Wind Power Density (National Average)', fontsize=16)
plt.xlabel('Month (1=Jan, 12=Dec)')
plt.ylabel('Wind Power Density (W/m^2)')
plt.yscale('log')  # Log scale to easily see distribution given extreme values
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation for a high-potential state (e.g., Gujarat)
state_data = df[df['State'] == 'Gujarat'].sort_values('Date').set_index('Date')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(state_data['Wind_Power_Density'], lags=30, ax=axes[0], title='Autocorrelation (ACF) - Gujarat')
plot_pacf(state_data['Wind_Power_Density'], lags=30, ax=axes[1], title='Partial Autocorrelation (PACF) - Gujarat')
plt.tight_layout()
plt.show()

## 3. Feature Correlations

In [ ]:
target_var = 'Wind_Power_Density'
ignore_vars = ['Date', 'State', 'Season', 'Month', 'Latitude', 'Longitude']
features = [col for col in df.columns if col not in ignore_vars]

plt.figure(figsize=(12, 10))
corr_matrix = df[features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Meteorological Features', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
print("Top Positive Correlations with Wind Power Density:")
print(corr_matrix[target_var].sort_values(ascending=False)[1:6])
print("\nTop Negative Correlations with Wind Power Density:")
print(corr_matrix[target_var].sort_values(ascending=True)[:5])